# XTTS-v2 fine-tune &mdash; VoiceMakers Sinhala female voices

Dinithi (4.82 h) + Harini (2.14 h) &asymp; **7 h across two distinct female speakers**.

| Setting | Value |
|---|---|
| Accelerator | **GPU P100** or **GPU T4 x2** (only one GPU is used) |
| Internet | **ON** |
| Input | `SinhalaTTS_Dataset_Publication_by_VoiceMakers` (already attached) |
| Persistence | **Files** &mdash; needed to resume past the 12 h session limit |

## Two things this run does differently

**The text becomes ASCII before it reaches the tokenizer.** XTTS-v2's `vocab.json` is a
whitespace-pretokenised BPE with an `[UNK]` fallback and contains no Sinhala codepoint &mdash;
nor the diacritics this corpus romanises with (`ā ī ū ē ṭ ḍ ṇ ḷ ṁ` are all missing). Fed
either column raw, **every word becomes one `[UNK]`** and the model trains on "unknown
unknown unknown": loss falls, audio is noise. Cell 4 asserts 0 `[UNK]` before any GPU time
is spent.

**Two speakers, correctly labelled.** XTTS samples a conditioning clip from the *same
speaker* on every step. Two real labels teach "the reference predicts the voice"; pooling
them under one label teaches the opposite, and no amount of data repairs it.

## 1. Install &mdash; restart the session after this cell

In [ ]:
# coqui-tts is the maintained idiap fork. Do NOT `pip install TTS` -- that one
# pins torch<2.1 and replaces Kaggle's CUDA build with a CPU wheel.
!pip install -q "coqui-tts>=0.25.1" "coqui-tts-trainer>=0.2.0" soundfile librosa tensorboard

import os
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else
      "\n*** NO GPU. Settings -> Accelerator -> GPU P100, then restart. ***")

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "").lower() == "batch":
    # Save & Run All cannot restart the session, and does not need to: every step
    # that imports TTS runs in a fresh subprocess and so picks these packages up
    # regardless of what this notebook process already imported.
    print("\nBatch mode -- no restart needed, continuing straight on.")
else:
    print("\n>>> Now: Run -> Restart session, then continue from the NEXT cell. <<<")


## 2. Code and paths

In [ ]:
import os, subprocess, pathlib

REPO = "https://github.com/DSEgrp18/Dataset-creation-withEmotion.git"
CODE = "/kaggle/working/Dataset-creation-withEmotion"
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=False)
SRC = CODE + "/xtts_model_female"

# Find the attached dataset wherever Kaggle mounted it -- the slug differs
# between "Add Data" and a notebook output, so search instead of hardcoding.
inp = pathlib.Path("/kaggle/input")
cands = [d for d in inp.iterdir() if d.is_dir()] if inp.is_dir() else []
DATA = None
for d in cands:
    names = {p.name.lower() for p in d.rglob("*") if p.is_dir()}
    if any("dinithi" in n for n in names) and any("harini" in n for n in names):
        DATA = str(d); break
print("code   :", SRC)
print("data   :", DATA or f"NOT FOUND -- dirs present: {[d.name for d in cands]}")

DATASET = "/kaggle/temp/female_dataset"     # scratch, not against the 20 GB quota
RUN     = "/kaggle/working/run"
EVAL    = "/kaggle/working/eval_out"
BASE    = RUN + "/training/XTTS_v2.0_original_model_files"
!mkdir -p /kaggle/temp

## 3. Inspect the layout before trusting it

The published folders are inconsistent (`Isuru-44100Hz` vs `Yasindu-44100`, and at least
one speaker directory nested inside a duplicate of itself). Look at what is actually there.

In [ ]:
import pathlib, collections
root = pathlib.Path(DATA)
for d in sorted(p for p in root.rglob("*") if p.is_dir()):
    wavs = list(d.glob("*.wav"))
    csvs = list(d.glob("*.csv"))
    if wavs or csvs:
        print(f"{str(d.relative_to(root)):45s} {len(wavs):5d} wav  {[c.name for c in csvs]}")

meta = sorted(root.rglob("metadata.csv"))
print("\nmetadata files:", [str(m.relative_to(root)) for m in meta])
if meta:
    print("\nfirst 3 raw lines of", meta[0].name)
    for line in meta[0].read_text(encoding="utf-8-sig").splitlines()[:3]:
        print("  ", line[:160])

## 4. Build the dataset &mdash; and prove the text tokenises

This **fails loudly** rather than training on garbage if the romanisation contains a
character `sinhala_text.py` does not map, or if any `[UNK]` survives.

In [ ]:
!wget -q -O /kaggle/temp/vocab.json https://huggingface.co/coqui/XTTS-v2/resolve/main/vocab.json
!cd {SRC} && python prepare_voicemakers.py \
    --src {DATA} --out {DATASET} --speakers dinithi harini \
    --vocab /kaggle/temp/vocab.json --eval-per-speaker 40

## 5. Smoke test &mdash; two minutes, catches every wiring fault

In [ ]:
!cd {SRC} && python train_xtts_female.py --dataset {DATASET} \
    --out /kaggle/working/smoke --smoke --batch-size 2 --grad-accum 2

# Look for a finite loss_mel_ce. `nan` means fp16 blew up -- add
# --no-mixed-precision to the real training cell below.

In [ ]:
!rm -rf /kaggle/working/smoke
!df -h /kaggle/working /kaggle/temp | grep -v Filesystem

## 6. Train

Backgrounded so the notebook stays responsive. Effective batch is
`batch_size x grad_accum = 64`. Upstream recommends 252, which is right for a datacentre;
on one T4 that is ~100 s per optimiser step and a whole session buys ~400 steps &mdash; far
too few to move the model onto a new sound inventory. Drop `--batch-size` to 3 or 2 on OOM
and raise `--grad-accum` to keep the product near 64.

In [ ]:
import os, subprocess, time

# Kaggle sets this to "Batch" under Save & Run All, "Interactive" otherwise.
BATCH = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive").lower() == "batch"
TRAIN_BUDGET_H = 8.5   # leave room for evaluation inside Kaggle's 12 h session cap

LOG = "/kaggle/working/train.log"
cmd = ["python", "train_xtts_female.py",
       "--dataset", DATASET, "--out", RUN,
       "--epochs", "40", "--batch-size", "4", "--grad-accum", "16",
       "--lr", "1e-5", "--save-step", "1000"]
print(" ".join(cmd))
print("mode:", "BATCH (blocking)" if BATCH else "INTERACTIVE (backgrounded)")

with open(LOG, "w") as fh:
    proc = subprocess.Popen(cmd, cwd=SRC, stdout=fh, stderr=subprocess.STDOUT)

if not BATCH:
    # Return now so the notebook stays usable. Follow it with the next cell,
    # re-running that cell whenever you want an update.
    print("pid", proc.pid, "-- training in the background")
    print("Re-run the next cell to watch the loss.")
else:
    # Save & Run All: every cell below needs a checkpoint to exist, so this has
    # to block. The wall-clock cap matters because Kaggle kills the session at
    # 12 h without warning, and a session killed mid-training leaves no time to
    # evaluate or export. Checkpoints are written every save_step and
    # best_model.pth at every eval, so a capped run is still fully usable -- it
    # has simply seen fewer epochs.
    deadline = time.time() + TRAIN_BUDGET_H * 3600
    last_beat = 0.0
    while proc.poll() is None and time.time() < deadline:
        time.sleep(30)
        if time.time() - last_beat > 600:          # heartbeat every ~10 min
            last_beat = time.time()
            try:
                with open(LOG, encoding="utf-8", errors="replace") as fh:
                    tail = [l.rstrip() for l in fh.readlines() if l.strip()][-2:]
                left = (deadline - time.time()) / 3600
                print(f"[{left:5.2f} h left] " + " | ".join(t[-160:] for t in tail),
                      flush=True)
            except Exception:
                pass
    if proc.poll() is None:
        print(f"\nBudget of {TRAIN_BUDGET_H} h reached -- stopping training here so "
              "evaluation and export still get to run.", flush=True)
        proc.terminate()
        try:
            proc.wait(timeout=300)
        except subprocess.TimeoutExpired:
            proc.kill()
    print("\ntraining process exited with code", proc.returncode)


In [ ]:
# Re-run to follow along. loss_mel_ce is the acoustic reconstruction term and the
# only one that tracks audio quality; loss_text_ce carries weight 0.01.
!grep -E "loss_mel_ce|EPOCH|EVAL|BEST" /kaggle/working/train.log | tail -n 25

### Resuming after the 12 h limit
Turn on **Persistence &rarr; Files**. Next session, re-run cells 1&ndash;4 then this instead of
the training cell above.

In [ ]:
# import glob, os
# prev = max(glob.glob(RUN + "/training/GPT_XTTS_si_female-*"), key=os.path.getmtime)
# !cd {SRC} && python train_xtts_female.py --dataset {DATASET} --out {RUN} \
#     --epochs 40 --batch-size 4 --grad-accum 16 --lr 1e-5 --continue-path {prev}

## 7. Objective evaluation

MCD, log-F0 RMSE, F0 correlation, speaker similarity, duration ratio, generation failure
rate and RTF over the held-out split. Add `--utmos` for the learned MOS predictor, and
`--asr openai/whisper-large-v3` for the CER gap (slow, large download).

In [ ]:
import glob, os
run = max(glob.glob(RUN + "/training/GPT_XTTS_si_female-*"), key=os.path.getmtime)
print("run:", run)

!cd {SRC} && python evaluate_xtts.py --run {run} --base {BASE} \
    --dataset {DATASET} --out {EVAL} --n 40 --utmos

In [ ]:
from IPython.display import Audio, Markdown, display
import glob, json

display(Markdown(open(EVAL + "/report.md", encoding="utf-8").read()))

ref = json.load(open(DATASET + "/eval_reference.json", encoding="utf-8"))
for it in ref[:4]:
    syn = EVAL + "/synth/" + it["clip_id"] + ".wav"
    if not glob.glob(syn):
        continue
    print("\n" + it["sinhala"])
    print("  speaker:", it["speaker"])
    print("  REAL recording:");  display(Audio(DATASET + "/" + it["wav"]))
    print("  SYNTHESISED:");     display(Audio(syn))

## 8. Build the MOS / SUS listening panel

The two metrics the literature actually compares on need human ears. This writes one
self-contained HTML file &mdash; download it from the output pane and send it to native
speakers; they rate in a browser and send back a CSV.

In [ ]:
!cd {SRC} && python listening_test.py --run {run} --base {BASE} \
    --dataset {DATASET} --out /kaggle/working/listening_test
!ls -la /kaggle/working/listening_test

## 9. Export the model

`model.pth` + `config.json` + `vocab.json` in one folder. Always run text through
`sinhala_text.to_ascii()` before synthesising &mdash; raw Sinhala gives `[UNK]` and noise.

In [ ]:
import shutil, os, glob
EXP = "/kaggle/working/xtts_si_female"
os.makedirs(EXP, exist_ok=True)
ck = run + "/best_model.pth"
if not os.path.isfile(ck):
    ck = max(glob.glob(run + "/checkpoint_*.pth"),
             key=lambda p: int(p.split("_")[-1].split(".")[0]))
shutil.copy2(ck, EXP + "/model.pth")
for f in ("config.json", "vocab.json"):
    shutil.copy2(BASE + "/" + f, EXP + "/" + f)
shutil.copy2(CODE + "/xtts_sinhala/sinhala_text.py", EXP + "/sinhala_text.py")
print("exported from", ck)
!du -sh {EXP} && ls -la {EXP}